# 🧠 Stage 1: Benchmark Ingestion & Verification (Clean Architecture)
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada  

---

### 🎯 Objectives:
1. Ingest **L0 to L5** benchmark datasets using the decoupled `HuggingFaceBenchmarkLoader`.
2. Standardize all problem schemas into domain `BenchmarkTask` entities.
3. Execute sandbox verification on canonical ground-truth solutions via `DataService`.
4. Inspect samples across all 6 ladder levels.

In [1]:
import os
import sys
import pandas as pd

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath(".."))

from src.services.data_service import DataService
from src.infrastructure.hf_loader import HuggingFaceBenchmarkLoader, LADDER_DATASET_CONFIGS
from src.infrastructure.sandbox import MultiprocessSandbox
from src.core.entities import BenchmarkTask

data_service = DataService(loader=HuggingFaceBenchmarkLoader(cache_dir="../data/ladder"))
print("✅ Clean Architecture DataService initialized successfully!")

c:\Users\Lenovo\anaconda3\envs\unsloth_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


✅ Clean Architecture DataService initialized successfully!


## 1. Overview of the Reduction Ladder Levels
Let's inspect the target benchmarks and their Hugging Face sources.

In [2]:
ladder_df = pd.DataFrame.from_dict(LADDER_DATASET_CONFIGS, orient="index")
ladder_df.index.name = "Level"
ladder_df

,name,dataset_path,split,trust_remote_code
Level,,,,
L0,HumanEval_Standard,openai/openai_humaneval,test,False
L1,EvoEval_Subtle,evoeval/EvoEval_subtle,test,False
L2,EvoEval_ToolUse,evoeval/EvoEval_tool_use,test,False
L3,EvoEval_Creative,evoeval/EvoEval_creative,test,False
L4,EvoEval_Difficult,evoeval/EvoEval_difficult,test,False
L5,EvoEval_Combine,evoeval/EvoEval_combine,test,False
Ctrl,LiveCodeBench_Lite,livecodebench/code_generation_lite,test,True


## 2. Ingest and Normalize All Ladder Levels (L0 to L5)

In [3]:
all_ladder_data = data_service.prepare_all_benchmarks(force_download=False)

print(f"\n🎉 All {len(all_ladder_data)} levels loaded as domain entities!")


🎉 All 6 levels loaded as domain entities!


## 3. Ground-Truth Sandbox Verification Suite
Run unit tests on all canonical solutions to ensure 100% test-suite correctness.

In [4]:
verification_results = {}
for level_key, tasks in all_ladder_data.items():
    print(f"\n==================== Verifying {level_key} ====================")
    res = data_service.verify_ground_truth(tasks)
    verification_results[level_key] = res

summary_df = pd.DataFrame([
    {"Level": k, "Total Tasks": v["total"], "Passed": v["passed"], "Pass Rate (%)": v["pass_rate"]}
    for k, v in verification_results.items()
])
summary_df


==================== Verifying L0 ====================


Verifying L0: 100%|██████████| 164/164 [00:01<00:00, 103.94it/s]



==================== Verifying L1 ====================


Verifying L1: 100%|██████████| 100/100 [00:04<00:00, 23.35it/s]



==================== Verifying L2 ====================


...


==================== Verifying L4 ====================


Verifying L4: 100%|██████████| 100/100 [00:06<00:00, 16.29it/s]



==================== Verifying L5 ====================


Verifying L5: 100%|██████████| 100/100 [00:03<00:00, 29.20it/s]


,Level,Total Tasks,Passed,Pass Rate (%)
0,L0,164,164,100.0
1,L1,100,99,99.0
2,L2,100,99,99.0
3,L3,100,100,100.0
4,L4,100,98,98.0
5,L5,100,97,97.0


## 4. Visual Inspection: Comparing Problem Transformations Across Levels

In [5]:
for level in ["L0", "L1", "L2", "L3", "L4", "L5"]:
    task = all_ladder_data[level][0]
    print(f"\n{'='*30} [{level}]: {task.benchmark} {'='*30}")
    print(f"Task ID: {task.task_id}")
    print("Prompt:")
    print(task.prompt[:300] + "...\n")


============================== [L0]: HumanEval_Standard ==============================
Task ID: HumanEval/0
Prompt:
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, ...


============================== [L1]: EvoEval_Subtle ==============================
Task ID: EvoEval/0
Prompt:

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two adjacent numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_eleme...


============================== [L2]: EvoEval_ToolUse ==============================
Task ID: EvoEval/0
Prompt:
def get_digit_count(n: int) -> int:
    return len